In [2]:
#Here is a basic implementation using the MarianMT model for translation (for example, English to French):

In [3]:
# !pip install transformers datasets

References:
https://github.com/Mooler0410/LLMsPracticalGuide 

In [4]:
from transformers import MarianMTModel, MarianTokenizer
from datasets import load_dataset

# Load the pre-trained model and tokenizer for English to French translation
model_name = 'Helsinki-NLP/opus-mt-en-fr'
tokenizer = MarianTokenizer.from_pretrained(model_name)
model = MarianMTModel.from_pretrained(model_name)


tokenizer_config.json:   0%|          | 0.00/42.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/778k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/802k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.34M [00:00<?, ?B/s]

/opt/anaconda3/lib/python3.11/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


config.json:   0%|          | 0.00/1.42k [00:00<?, ?B/s]

/opt/anaconda3/lib/python3.11/site-packages/transformers/models/marian/tokenization_marian.py:175: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


pytorch_model.bin:   0%|          | 0.00/301M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

In [5]:
# Load a small dataset (Here we use a sample dataset from Hugging Face or create your own dataset)
# For simplicity, let's create a simple dataset
data = [
    {"translation": {"en": "Hello, how are you?", "fr": "Bonjour, comment ça va?"}},
    {"translation": {"en": "I love programming.", "fr": "J'aime programmer."}},
    {"translation": {"en": "This is a machine translation example.", "fr": "Ceci est un exemple de traduction automatique."}},
]



In [6]:
# Tokenize the input sentences (English in this case)
def encode_sentences(sentences, tokenizer, max_length=40):
    tokens = tokenizer(sentences, return_tensors="pt", padding=True, truncation=True, max_length=max_length)
    return tokens



In [7]:
# Translate function
def translate(sentences, model, tokenizer):
    # Tokenize input
    tokenized_input = encode_sentences(sentences, tokenizer)

    # Perform translation
    translated_tokens = model.generate(**tokenized_input)

    # Decode the translation into readable text
    translated_sentences = [tokenizer.decode(t, skip_special_tokens=True) for t in translated_tokens]

    return translated_sentences



In [8]:
# Example sentences for translation (English to French)
source_sentences = [
    "Hello, how are you?",
    "I love programming.",
    "This is a machine translation example."
]

# Translate the sentences
translated_sentences = translate(source_sentences, model, tokenizer)

# Print results
for src, tgt in zip(source_sentences, translated_sentences):
    print(f"Source: {src} -> Translated: {tgt}")


Source: Hello, how are you? -> Translated: Bonjour, comment allez-vous?
Source: I love programming. -> Translated: J'adore la programmation.
Source: This is a machine translation example. -> Translated: C'est un exemple de traduction automatique.


# **Large Dataset**

In [9]:
from datasets import load_dataset

from datasets import load_dataset

# Load the dataset
dataset = load_dataset("opus_books", "en-fr")

# Use select() to get the first 10 items from the dataset
subset = dataset['train'].select(range(10))

# Prepare English and French sentences
en_sentences = [item['translation']['en'] for item in subset]
fr_sentences = [item['translation']['fr'] for item in subset]

# Print the sentences to verify
print("English Sentences:", en_sentences)
print("French Sentences:", fr_sentences)


README.md:   0%|          | 0.00/28.1k [00:00<?, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/127085 [00:00<?, ? examples/s]

English Sentences: ['The Wanderer', 'Alain-Fournier', 'First Part', 'I', 'THE BOARDER', 'He arrived at our home on a Sunday of November, 189-.', "I still say 'our home,' although the house no longer belongs to us.", 'We left that part of the country nearly fifteen years ago and shall certainly never go back to it.', "We were living in the building of the Higher Elementary Classes at Sainte-Agathe's School.", "My father, whom I used to call M. Seurel as did other pupils, was head of the Middle School and also of the Higher Elementary classes where pupils worked for the preliminary teacher's examination."]
French Sentences: ['Le grand Meaulnes', 'Alain-Fournier', 'PREMIÈRE PARTIE', 'CHAPITRE PREMIER', 'LE PENSIONNAIRE', 'Il arriva chez nous un dimanche de novembre 189-…', 'Je continue à dire « chez nous », bien que la maison ne nous appartienne plus.', 'Nous avons quitté le pays depuis bientôt quinze ans et nous n’y reviendrons certainement jamais.', 'Nous habitions les bâtiments du Cour

In [10]:
print(dataset['train'][0])  # This will print the structure of one item in the dataset


{'id': '0', 'translation': {'en': 'The Wanderer', 'fr': 'Le grand Meaulnes'}}


In [11]:

# Translate English to French
translated_fr_sentences = translate(en_sentences, model, tokenizer)

# Print some sample translations
for en, fr, pred_fr in zip(en_sentences, fr_sentences, translated_fr_sentences):
    print(f"Source: {en} -> Target: {fr} -> Predicted: {pred_fr}")


Source: The Wanderer -> Target: Le grand Meaulnes -> Predicted: Le Wanderer
Source: Alain-Fournier -> Target: Alain-Fournier -> Predicted: Alain-Fournier
Source: First Part -> Target: PREMIÈRE PARTIE -> Predicted: Première partie
Source: I -> Target: CHAPITRE PREMIER -> Predicted: Annexe I
Source: THE BOARDER -> Target: LE PENSIONNAIRE -> Predicted: LE CONSEIL D'ADMINISTRATION
Source: He arrived at our home on a Sunday of November, 189-. -> Target: Il arriva chez nous un dimanche de novembre 189-… -> Predicted: Il est arrivé à notre maison un dimanche de Novembre, 189-.
Source: I still say 'our home,' although the house no longer belongs to us. -> Target: Je continue à dire « chez nous », bien que la maison ne nous appartienne plus. -> Predicted: Je dis toujours "notre maison", bien que la maison ne nous appartienne plus.
Source: We left that part of the country nearly fifteen years ago and shall certainly never go back to it. -> Target: Nous avons quitté le pays depuis bientôt quinze 



---



In [12]:
from datasets import load_dataset

# Load a dataset for English-French translation (e.g., 'opus_books')
dataset = load_dataset("opus_books", "en-fr")

# Select a small subset of the data for demonstration
#train_data = dataset['train'].select(range(100))  # Use a small subset for training


In [13]:
# Split the dataset into train and eval datasets
split_dataset = dataset['train'].select(range(100)).train_test_split(test_size=0.1)
train_data = split_dataset['train']
eval_data = split_dataset['test']


In [14]:
from transformers import MarianMTModel, MarianTokenizer, Seq2SeqTrainer, Seq2SeqTrainingArguments, DataCollatorForSeq2Seq

# Load the pre-trained MarianMT model and tokenizer
model_name = 'Helsinki-NLP/opus-mt-en-fr'
tokenizer = MarianTokenizer.from_pretrained(model_name)
model = MarianMTModel.from_pretrained(model_name)


2024-09-17 11:44:01.159794: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/opt/anaconda3/lib/python3.11/site-packages/transformers/models/marian/tokenization_marian.py:175: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


In [15]:
# Tokenize the dataset
def preprocess_function(examples):
    inputs = [ex['en'] for ex in examples['translation']]
    targets = [ex['fr'] for ex in examples['translation']]
    model_inputs = tokenizer(inputs, text_target=targets, max_length=128, truncation=True)
    return model_inputs

# Tokenize dataset
tokenized_train_data = train_data.map(preprocess_function, batched=True)
tokenized_eval_data = eval_data.map(preprocess_function, batched=True)


Map:   0%|          | 0/90 [00:00<?, ? examples/s]

Map:   0%|          | 0/10 [00:00<?, ? examples/s]

In [16]:
# Define training arguments
training_args = Seq2SeqTrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",  # Enable evaluation at the end of every epoch
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    save_total_limit=3,
    predict_with_generate=True
)

/opt/anaconda3/lib/python3.11/site-packages/transformers/training_args.py:1474: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [17]:
# Data collator for padding
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)



In [18]:
# Define the trainer
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train_data,  # Training dataset
    eval_dataset=tokenized_eval_data,    # Evaluation dataset
    data_collator=data_collator,
    tokenizer=tokenizer
)


In [19]:

# Fine-tune the model
trainer.train()


  0%|          | 0/18 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

{'eval_loss': 1.9358417987823486, 'eval_runtime': 1.2135, 'eval_samples_per_second': 8.241, 'eval_steps_per_second': 0.824, 'epoch': 1.0}


  0%|          | 0/1 [00:00<?, ?it/s]

{'eval_loss': 1.896483063697815, 'eval_runtime': 1.1601, 'eval_samples_per_second': 8.62, 'eval_steps_per_second': 0.862, 'epoch': 2.0}


  0%|          | 0/1 [00:00<?, ?it/s]

{'eval_loss': 1.885469675064087, 'eval_runtime': 1.3827, 'eval_samples_per_second': 7.232, 'eval_steps_per_second': 0.723, 'epoch': 3.0}
{'train_runtime': 128.0791, 'train_samples_per_second': 2.108, 'train_steps_per_second': 0.141, 'train_loss': 2.1994427575005426, 'epoch': 3.0}


TrainOutput(global_step=18, training_loss=2.1994427575005426, metrics={'train_runtime': 128.0791, 'train_samples_per_second': 2.108, 'train_steps_per_second': 0.141, 'total_flos': 6849058701312.0, 'train_loss': 2.1994427575005426, 'epoch': 3.0})

In [24]:
# Save the fine-tuned model and tokenizer
model.save_pretrained("./temp/my_custom_marian_mt")
tokenizer.save_pretrained("./temp/my_custom_marian_mt")


Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model) instead. This warning will be raised to an exception in v4.41.
Non-default generation parameters: {'max_length': 512, 'num_beams': 4, 'bad_words_ids': [[59513]], 'forced_eos_token_id': 0}


('./temp/my_custom_marian_mt/tokenizer_config.json',
 './temp/my_custom_marian_mt/special_tokens_map.json',
 './temp/my_custom_marian_mt/vocab.json',
 './temp/my_custom_marian_mt/source.spm',
 './temp/my_custom_marian_mt/target.spm',
 './temp/my_custom_marian_mt/added_tokens.json')

In [26]:
from transformers import MarianMTModel, MarianTokenizer

# Load your fine-tuned model and tokenizer
model = MarianMTModel.from_pretrained("./temp/my_custom_marian_mt")
tokenizer = MarianTokenizer.from_pretrained("./temp/my_custom_marian_mt")


/opt/anaconda3/lib/python3.11/site-packages/transformers/models/marian/tokenization_marian.py:175: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


In [27]:
!zip -r ./models/eng_french_lang_translator_marian_mt.zip ./temp/my_custom_marian_mt/


  adding: temp/my_custom_marian_mt/ (stored 0%)
  adding: temp/my_custom_marian_mt/model.safetensors (deflated 7%)
  adding: temp/my_custom_marian_mt/target.spm (deflated 50%)
  adding: temp/my_custom_marian_mt/tokenizer_config.json (deflated 68%)
  adding: temp/my_custom_marian_mt/special_tokens_map.json (deflated 73%)
  adding: temp/my_custom_marian_mt/config.json (deflated 61%)
  adding: temp/my_custom_marian_mt/generation_config.json (deflated 43%)
  adding: temp/my_custom_marian_mt/source.spm (deflated 49%)
  adding: temp/my_custom_marian_mt/vocab.json (deflated 70%)


Simple Application Test

In [ ]:

# Function to translate a sentence from English to French
def translate(sentence):
    inputs = tokenizer(sentence, return_tensors="pt", padding=True, truncation=True)
    translated_tokens = model.generate(**inputs)
    return tokenizer.decode(translated_tokens[0], skip_special_tokens=True)

# User input loop for translation
while True:
    # Get user input
    english_sentence = input("Enter an English sentence (or 'exit' to quit): ")

    # Check if the user wants to exit
    if english_sentence.lower() == 'exit':
        print("Exiting the translation system.")
        break

    # Translate the sentence
    french_translation = translate(english_sentence)

    # Output the translated sentence
    print(f"French Translation: {french_translation}\n")


### Mistral is a good open-source model (encoder-decoder architecture)

Reference:
https://huggingface.co/mistralai/Mistral-7B-Instruct-v0.1

In [ ]:
# # Load model directly
# from transformers import AutoTokenizer, AutoModelForCausalLM

# tokenizer = AutoTokenizer.from_pretrained("mistralai/Mistral-7B-Instruct-v0.1")
# model = AutoModelForCausalLM.from_pretrained("mistralai/Mistral-7B-Instruct-v0.1")